In [2]:
import pickle
import os
import numpy as np
import json
import kagglehub
import pandas as pd


DATA_DIR = os.path.join("..", "data")



In [3]:
with open(os.path.join(DATA_DIR, "als_model.pkl"), "rb") as f:
    model = pickle.load(f)
with open(os.path.join(DATA_DIR, "anilist_to_mal.json"), "r") as f:
    mal_ids = json.load(f)
with open(os.path.join(DATA_DIR, "anime_data.jsonl"), "r") as f:
    anime_content = [json.loads(line) for line in f]

train = pd.read_parquet(os.path.join(DATA_DIR, "train_ratings.parquet"))

path = kagglehub.dataset_download("ramazanturann/user-animelist-dataset")
animes = pd.read_csv(os.path.join(path, "animes.csv"))

als_user_factors = model.user_factors  # shape: (num_users, 64)
als_item_factors = model.item_factors  # shape: (num_anime, 64)

print(als_user_factors.shape, als_item_factors.shape)

(1774060, 64) (19903, 64)


In [4]:
anime_tag_vectors = np.load(os.path.join(DATA_DIR, "anime_tag_vectors.npy"))
manga_tag_vectors = np.load(os.path.join(DATA_DIR, "manga_tag_vectors.npy"))

print(anime_tag_vectors.shape, manga_tag_vectors.shape)

(5000, 422) (5000, 422)


In [5]:
anime_ids = train['anime_id'].astype('category')
anime_id_map = dict(enumerate(anime_ids.cat.categories))
anime_id_map_reverse = {v: k for k, v in anime_id_map.items()}

# AniList ID -> real MAL ID (via idMal crosswalk; drop failed lookups)
anilist_to_mal = {int(k): v for k, v in mal_ids.items() if v is not None}

none_count = sum(1 for v in mal_ids.values() if v is None)
print(f"anilist_to_mal: {none_count}/{len(mal_ids)} AniList entries had no MAL match (idMal was null) -- dropped")

# real MAL ID -> ratings dataset's internal animeID (bridge step that was missing before)
animes['mal_id'] = animes['mal_url'].str.extract(r'/anime/(\d+)').astype(int)

# --- sanity-check the bridge tables before trusting a dict built from them ---
dup_mal = animes['mal_id'].duplicated(keep=False)
dup_animeid = animes['animeID'].duplicated(keep=False)
if dup_mal.any():
    print(f"WARNING: {dup_mal.sum()} rows in animes share a duplicated mal_id -- "
          f"dict(zip(...)) will silently keep only the last row per key")
if dup_animeid.any():
    print(f"WARNING: {dup_animeid.sum()} rows in animes share a duplicated animeID")
if not dup_mal.any() and not dup_animeid.any():
    print("animes['mal_id'] and animes['animeID'] are both unique")

mal_to_animeid = dict(zip(animes['mal_id'], animes['animeID']))

# --- walk the full chain: AniList idx -> real MAL id -> dataset animeID -> ALS row ---
aligned_rows = []
for i, a in enumerate(anime_content):
    anilist_id = a['id']
    real_mal_id = anilist_to_mal.get(anilist_id)
    if real_mal_id is None:
        continue  # no MAL match for this AniList entry


    animeid = mal_to_animeid.get(real_mal_id)
    if animeid is None:
        continue  # MAL id doesn't appear in the ratings dataset at all

    als_row = anime_id_map_reverse.get(animeid)
    if als_row is None:
        continue  # in the ratings dataset, but never rated in `train` -> no ALS row

    aligned_rows.append({
        'anilist_idx': i,
        'real_mal_id': real_mal_id,
        'animeid': animeid,
        'als_row': als_row,
    })

aligned_df = pd.DataFrame(aligned_rows)
print(f"Aligned {len(aligned_df)} / {len(anime_content)} AniList anime through all three ID systems to an ALS row")


anilist_to_mal: 11/4950 AniList entries had no MAL match (idMal was null) -- dropped
animes['mal_id'] and animes['animeID'] are both unique -- safe to use as dict keys
Aligned 4778 / 5000 AniList anime through all three ID systems to an ALS row


In [10]:
def anilist_title(a):
    # AniList title is a dict, not a plain string
    t = a['title']
    return t.get('english') or t.get('romaji') or t.get('native')

animes_title_by_id = animes.set_index('animeID')['title']

sample = aligned_df.sample(min(10, len(aligned_df)), random_state=1)
for _, row in sample.iterrows():
    anilist_t = anilist_title(anime_content[row['anilist_idx']])
    ratings_t = animes_title_by_id.loc[row['animeid']]
    print(f"AniList: {anilist_t!r:55} | animes: {ratings_t!r}")


AniList: 'Granblue Fantasy: The Animation Season 2'              | animes: 'Granblue Fantasy: The Animation Season 2'
AniList: 'The Legend of Hei'                                     | animes: 'The Legend of Hei'
AniList: 'LAID-BACK CAMP SEASON2'                                | animes: 'Laid-Back Camp Season 2'
AniList: 'Hokkaido Gals Are Super Adorable!'                     | animes: 'Hokkaido Gals Are Super Adorable!'
AniList: 'Asteroid in Love'                                      | animes: 'Asteroid in Love'
AniList: 'Major S3'                                              | animes: 'Major S3'
AniList: 'Mobile Suit Gundam 00 Second Season'                   | animes: 'Mobile Suit Gundam 00: Second Season'
AniList: 'Gintama.: Silver Soul Arc'                             | animes: 'Gintama. Silver Soul Arc'
AniList: 'Blue Miburo'                                           | animes: 'Blue Miburo'
AniList: 'Pupa'                                                  | animes: 'Pupa'
